# Two Pointers, Sliding Window and Prefix Sums

Notebook for **Chapter 3** of the *dsa* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dsa/ch03.html

Tip: change a number, predict the output, then run the cell again.

## 3.1 The Big Picture: Why These Tricks Work

In [ ]:
a = list(range(1000))
target = -5                       # impossible, so every method must do its full work

pairs = 0
for i in range(len(a)):
    for j in range(i + 1, len(a)):
        pairs += 1                # brute force looks at every pair

steps, l, r = 0, 0, len(a) - 1
while l < r:                      # two pointers
    steps += 1
    if a[l] + a[r] < target:
        l += 1
    else:
        r -= 1
print("brute force pairs:", pairs)
print("two pointer steps:", steps)
assert pairs == 499500 and steps == 999

brute force pairs: 499500
two pointer steps: 999


## 3.2 Opposite-End Two Pointers

In [ ]:
def two_sum_sorted(a, target):
    l, r = 0, len(a) - 1             # one pointer at each end
    while l < r:                     # l == r would be the SAME item
        s = a[l] + a[r]              # CHANGE: the value you inspect
        if s == target:              # CHANGE: the "found" test
            return (l, r)
        if s < target:
            l += 1                   # too small: the left item is useless
        else:
            r -= 1                   # too big: the right item is useless
    return None

a = [1, 3, 4, 6, 8, 11, 15]
print(two_sum_sorted(a, 14), two_sum_sorted(a, 100))
assert two_sum_sorted(a, 14) == (1, 5) and two_sum_sorted([7], 7) is None

(1, 5) None


In [ ]:
def max_water(h):
    l, r, best = 0, len(h) - 1, 0
    while l < r:
        best = max(best, min(h[l], h[r]) * (r - l))
        if h[l] < h[r]:
            l += 1                   # move the SHORTER wall
        else:
            r -= 1
    return best

h = [3, 9, 2, 6, 8, 4, 7, 5]
print(max_water(h))
assert max_water(h) == max(min(h[i], h[j]) * (j - i) for i in range(8) for j in range(i + 1, 8)) == 35

35


In [ ]:
def three_sum(nums):
    a, out = sorted(nums), []
    for i in range(len(a) - 2):
        if i > 0 and a[i] == a[i - 1]:       # same first number as before: skip
            continue
        l, r = i + 1, len(a) - 1
        while l < r:
            s = a[i] + a[l] + a[r]
            if s < 0:
                l += 1
            elif s > 0:
                r -= 1
            else:
                out.append([a[i], a[l], a[r]])
                l += 1
                r -= 1
                while l < r and a[l] == a[l - 1]:   # skip repeated second numbers
                    l += 1
    return out

print(three_sum([-4, -1, -1, 0, 1, 2]))
assert three_sum([0, 0, 0, 0]) == [[0, 0, 0]] and three_sum([1, 2]) == []

[[-1, -1, 2], [-1, 0, 1]]


## 3.3 Same-Direction Pointers: Reader and Writer

In [ ]:
def compact(a, keep):
    w = 0                            # writer: next free slot
    for r in range(len(a)):          # reader: visits every item once
        if keep(a, r, w):
            a[w] = a[r]              # copy the good item forward
            w += 1
    return w                         # new length; a[:w] is the answer

a = [1, 1, 2, 2, 2, 3, 4, 4]         # sorted: keep an item if it differs from the last kept one
n = compact(a, lambda a, r, w: w == 0 or a[r] != a[w - 1])
print(n, a[:n])
assert n == 4 and a[:n] == [1, 2, 3, 4] and compact([], lambda a, r, w: True) == 0

4 [1, 2, 3, 4]


In [ ]:
def move_zeroes(a):
    w = 0
    for r in range(len(a)):
        if a[r] != 0:
            a[w], a[r] = a[r], a[w]
            w += 1

a = [0, 1, 0, 3, 12]
move_zeroes(a)
print(a)
assert a == [1, 3, 12, 0, 0]

[1, 3, 12, 0, 0]


## 3.4 Fast and Slow Pointers

In [ ]:
class Node:
    def __init__(self, val, next=None):
        self.val, self.next = val, next

def build(values, loop_to=None):
    nodes = [Node(v) for v in values]
    for a, b in zip(nodes, nodes[1:]):
        a.next = b
    if loop_to is not None:
        nodes[-1].next = nodes[loop_to]      # last node points back: a cycle
    return nodes[0]

def meeting_node(head):
    slow = fast = head
    while fast and fast.next:        # test fast first, then fast.next
        slow = slow.next             # 1 step
        fast = fast.next.next        # 2 steps
        print("slow", slow.val, "fast", fast.val)
        if slow is fast:
            return slow
    return None                      # fast fell off the end: no cycle

head = build([1, 2, 3, 4, 5, 6], loop_to=2)
print("met at", meeting_node(head).val)
assert meeting_node(build([1, 2, 3])) is None

slow 2 fast 3
slow 3 fast 5
slow 4 fast 3
slow 5 fast 5
met at 5
slow 2 fast 3


In [ ]:
class Node:
    def __init__(self, val, next=None):
        self.val, self.next = val, next

def loop_start(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:             # phase 1: met inside the loop
            slow = head              # phase 2: restart one pointer at the head
            while slow is not fast:
                slow, fast = slow.next, fast.next
            return slow
    return None

nodes = [Node(v) for v in [1, 2, 3, 4, 5, 6]]
for a, b in zip(nodes, nodes[1:]):
    a.next = b
nodes[-1].next = nodes[2]
print(loop_start(nodes[0]).val)
assert loop_start(nodes[0]).val == 3 and loop_start(Node(9)) is None

3


In [ ]:
def find_duplicate(nums):
    slow = fast = 0
    while True:                      # phase 1: meet inside the loop
        slow = nums[slow]
        fast = nums[nums[fast]]
        if slow == fast:
            break
    slow = 0                         # phase 2: walk from the start
    while slow != fast:
        slow = nums[slow]
        fast = nums[fast]
    return slow

print(find_duplicate([3, 1, 3, 4, 2]))
assert find_duplicate([3, 1, 3, 4, 2]) == 3 and find_duplicate([1, 1]) == 1

3


## 3.5 Fixed-Size Sliding Window

In [ ]:
def max_window_sum(a, k):
    window = sum(a[:k])              # first frame, the slow way, once
    best = window
    for r in range(k, len(a)):       # r: the item entering on the right
        window += a[r] - a[r - k]    # add the new item, subtract the one that left
        best = max(best, window)
        print("r =", r, "in", a[r], "out", a[r - k], "sum", window, "best", best)
    return best

a = [2, 1, 5, 1, 3, 2, 9, 1]
print(max_window_sum(a, 3))
assert max_window_sum(a, 3) == max(sum(a[i:i + 3]) for i in range(6)) == 14

r = 3 in 1 out 2 sum 7 best 8
r = 4 in 3 out 1 sum 9 best 9
r = 5 in 2 out 5 sum 6 best 9
r = 6 in 9 out 1 sum 14 best 14
r = 7 in 1 out 3 sum 12 best 14
14
r = 3 in 1 out 2 sum 7 best 8
r = 4 in 3 out 1 sum 9 best 9
r = 5 in 2 out 5 sum 6 best 9
r = 6 in 9 out 1 sum 14 best 14
r = 7 in 1 out 3 sum 12 best 14


In [ ]:
from collections import Counter

def find_anagrams(s, p):
    k = len(p)
    if k > len(s):
        return []
    need, window, out = Counter(p), Counter(s[:k]), []
    if window == need:
        out.append(0)
    for r in range(k, len(s)):
        window[s[r]] += 1            # letter enters
        left = s[r - k]
        window[left] -= 1            # letter leaves
        if window[left] == 0:
            del window[left]         # no zero counts, so == works on every Python
        if window == need:
            out.append(r - k + 1)
    return out

print(find_anagrams("cbaebabacd", "abc"))
assert find_anagrams("abab", "ab") == [0, 1, 2] and find_anagrams("a", "ab") == []

[0, 6]


## 3.6 Variable-Size Sliding Window

In [ ]:
def longest_unique(s):
    last = {}                        # letter -> index where it was last seen
    l = best = 0
    for r, ch in enumerate(s):       # r only moves forward
        if ch in last and last[ch] >= l:     # the repeat is INSIDE the window
            l = last[ch] + 1                 # jump just past the old copy
        last[ch] = r
        best = max(best, r - l + 1)
        print("r =", r, ch, "l =", l, "window", s[l:r + 1], "best", best)
    return best

print(longest_unique("abcdbea"))
assert longest_unique("") == 0 and longest_unique("abba") == 2

r = 0 a l = 0 window a best 1
r = 1 b l = 0 window ab best 2
r = 2 c l = 0 window abc best 3
r = 3 d l = 0 window abcd best 4
r = 4 b l = 2 window cdb best 4
r = 5 e l = 2 window cdbe best 4
r = 6 a l = 2 window cdbea best 5
5
r = 0 a l = 0 window a best 1
r = 1 b l = 0 window ab best 2
r = 2 b l = 2 window b best 2
r = 3 a l = 2 window ba best 2


In [ ]:
def shortest_at_least(a, target):
    l = total = 0
    best = float("inf")
    for r, x in enumerate(a):
        total += x
        while total >= target:       # valid: try to shrink
            best = min(best, r - l + 1)      # shortest => record INSIDE the loop
            total -= a[l]
            l += 1
    return 0 if best == float("inf") else best

print(shortest_at_least([2, 3, 1, 2, 4, 3], 7))
assert shortest_at_least([1, 1, 1], 10) == 0 and shortest_at_least([9], 7) == 1

2


In [ ]:
from collections import defaultdict

def longest_k_distinct(s, k):
    count = defaultdict(int)
    l = best = 0
    for r, ch in enumerate(s):
        count[ch] += 1
        while len(count) > k:        # too many kinds: shrink
            count[s[l]] -= 1
            if count[s[l]] == 0:
                del count[s[l]]
            l += 1
        best = max(best, r - l + 1)  # longest => record AFTER the loop
    return best

print(longest_k_distinct("eceba", 2))
assert longest_k_distinct("aa", 1) == 2 and longest_k_distinct("abc", 0) == 0

3


In [ ]:
from collections import Counter
def min_window(s, t):
    need, missing, l = Counter(t), len(t), 0
    best = (float("inf"), 0, 0)
    for r, ch in enumerate(s):
        if need[ch] > 0:
            missing -= 1             # this letter fills a required slot
        need[ch] -= 1
        if missing == 0:             # s[l..r] has everything
            while need[s[l]] < 0:    # drop spare letters on the left
                need[s[l]] += 1
                l += 1
            if r - l + 1 < best[0]:
                best = (r - l + 1, l, r + 1)
            need[s[l]] += 1          # give up s[l]: window invalid again
            missing += 1
            l += 1
    return "" if best[0] == float("inf") else s[best[1]:best[2]]

print(min_window("ADOBECODEBANC", "ABC"))
assert min_window("a", "aa") == "" and min_window("a", "a") == "a"

BANC


## 3.7 Prefix Sums and Difference Arrays

In [ ]:
from itertools import accumulate

a = [3, 1, 4, 1, 5, 9, 2, 6]
P = [0] + list(accumulate(a))        # P[0] = 0, P[i+1] = P[i] + a[i]
print(P)

def range_sum(P, l, r):              # a[l..r], both ends included
    return P[r + 1] - P[l]

print(range_sum(P, 2, 5))
assert range_sum(P, 2, 5) == sum(a[2:6]) == 19 and range_sum(P, 0, 7) == sum(a)

[0, 3, 4, 8, 9, 14, 23, 25, 31]
19


In [ ]:
def count_sum_k(a, k):
    seen = {0: 1}                     # the empty prefix has sum 0
    prefix = count = 0
    for x in a:
        prefix += x
        count += seen.get(prefix - k, 0)          # lookup first ...
        seen[prefix] = seen.get(prefix, 0) + 1    # ... then record
    return count

a = [2, 3, -1, 4, 1, 2]
print(count_sum_k(a, 5))
brute = sum(sum(a[i:j + 1]) == 5 for i in range(6) for j in range(i, 6))
assert count_sum_k(a, 5) == brute == 2 and count_sum_k([0, 0, 0], 0) == 6

2


In [ ]:
from itertools import accumulate

def apply_updates(n, updates):
    diff = [0] * (n + 1)              # spare slot so r + 1 is always valid
    for l, r, v in updates:
        diff[l] += v                  # extra value starts here
        diff[r + 1] -= v              # and stops right after r
    return list(accumulate(diff))[:n]

ups = [(1, 3, 5), (2, 4, 2), (0, 1, -1)]
print(apply_updates(6, ups))
brute = [0] * 6
for l, r, v in ups:
    for i in range(l, r + 1):
        brute[i] += v
assert apply_updates(6, ups) == brute == [-1, 4, 7, 7, 2, 0]

[-1, 4, 7, 7, 2, 0]


## 3.8 Two-Dimensional Prefix Sums

In [ ]:
def build_2d(g):
    R, C = len(g), len(g[0])
    P = [[0] * (C + 1) for _ in range(R + 1)]     # extra row and column of zeros
    for i in range(R):
        for j in range(C):
            P[i + 1][j + 1] = g[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
    return P

def region_sum(P, r1, c1, r2, c2):                 # both corners included
    return P[r2 + 1][c2 + 1] - P[r1][c2 + 1] - P[r2 + 1][c1] + P[r1][c1]

g = [[3, 0, 1, 4, 2], [5, 6, 3, 2, 1], [1, 2, 0, 1, 5], [4, 1, 0, 1, 7]]
P = build_2d(g)
print(P[2])
print(region_sum(P, 1, 1, 2, 3))
assert region_sum(P, 1, 1, 2, 3) == 6 + 3 + 2 + 2 + 0 + 1 == 14
assert region_sum(P, 0, 0, 3, 4) == sum(map(sum, g)) == 49

[0, 8, 14, 18, 24, 27]
14


## 3.9 Choosing the Pattern

Ask these questions from top to bottom and stop at the first yes.

## Chapter recap

## Practice set

Graded from easy to hard. Try each before reading the answer.

In [ ]:
a = [0, 1, 0, 2]
w = 0
for r in range(len(a)):
    if a[r] != 0:
        a[w] = a[r]
        w += 1
print(a, w)

[1, 2, 0, 2] 2


In [ ]:
def two_sum_sorted(a, target):
    l, r = 0, len(a) - 1
    while l <= r:
        s = a[l] + a[r]
        if s == target:
            return (l, r)
        if s < target:
            l += 1
        else:
            r -= 1
    return None

print(two_sum_sorted([5], 10))

(0, 0)


In [ ]:
def longest_unique(s):
    last, l, best = {}, 0, 0
    for r, ch in enumerate(s):
        if ch in last:
            l = last[ch] + 1
        last[ch] = r
        best = max(best, r - l + 1)
    return best

print(longest_unique("abba"))

3


## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dsa/ch03.html